# Day 16：Tuned XGBoost Official Test Evaluation

Day16 只对 Day15 在 valid 上选出的 tuned XGBoost 候选方案做 official test 最终观察。本轮固定 Day15 的参数和 threshold，不重新搜索、不重新调阈值，也不根据 official test 结果回头修改 Day15。

## 关键边界

- official test 只用于最终观察。
- 参数和阈值来自 Day15 valid best summary。
- imputer 和结构特征规则只在 train_inner 上 fit。
- test 只 transform 和评估。
- 如果 tuned 方案 test 没有明显提升，应停止继续追模型，转向 SQL 深化、解释性分析和 README 收尾。

In [5]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT / "src"))

import pandas as pd

from scania_aps.config import get_config
from scania_aps.data.load_data import load_train_test_with_target
from scania_aps.data.split_data import split_train_valid
from scania_aps.features.structural_feature_design import load_structural_feature_config
from scania_aps.models.xgb_tuning_test_evaluation import (
    DEFAULT_DAY16_CANDIDATE_STRATEGIES,
    evaluate_tuned_xgb_candidates_on_test,
)

cfg = get_config(PROJECT_ROOT / "config" / "config.yaml")
structural_config = load_structural_feature_config(PROJECT_ROOT / "config" / "structural_features.yaml")
day15_best_summary = pd.read_csv(cfg.metrics_dir / "day15_xgb_tuning_valid_best_summary.csv")
DEFAULT_DAY16_CANDIDATE_STRATEGIES

['median_all_structural_all',
 'drop_high_missing_median',
 'baseline_median_all']

## 读取 Day16 输出

建议优先运行脚本生成结果：

```powershell
python scripts/14_xgb_tuning_test_evaluation.py
```

如果输出已存在，notebook 直接读取，避免重复训练。

In [6]:
test_results_path = cfg.metrics_dir / "day16_xgb_tuning_test_results.csv"
compare_path = cfg.metrics_dir / "day16_xgb_tuning_valid_test_compare.csv"

if test_results_path.exists():
    test_results = pd.read_csv(test_results_path)
    valid_test_compare = pd.read_csv(compare_path)
else:
    train_df, test_df = load_train_test_with_target(cfg)
    train_inner_df, valid_df = split_train_valid(train_df, cfg)
    results = evaluate_tuned_xgb_candidates_on_test(
        train_inner_df=train_inner_df,
        valid_df=valid_df,
        test_df=test_df,
        cfg=cfg,
        structural_config=structural_config,
        day15_best_summary=day15_best_summary,
        candidate_strategies=DEFAULT_DAY16_CANDIDATE_STRATEGIES,
    )
    test_results = results["tuned_test_results"]
    valid_test_compare = results["tuned_valid_test_compare"]

display_cols = ["candidate_strategy", "threshold", "precision", "recall", "f2", "average_precision", "fp", "fn", "total_cost"]
test_results[display_cols]

,candidate_strategy,threshold,precision,recall,f2,average_precision,fp,fn,total_cost
0,drop_high_missing_median,0.19,0.489130,0.960000,0.805009,0.912119,376,15,11260
1,baseline_median_all,0.13,0.512857,0.957333,0.815909,0.921827,341,16,11410
2,median_all_structural_all,0.31,0.553822,0.946667,0.829052,0.913298,286,20,12860


## valid vs official test

这里观察 Day15 valid 最优是否能泛化到 official test。test 结果不能用于反向修改 Day15 参数或阈值。

In [7]:
valid_test_compare[[
    "candidate_strategy", "threshold", "valid_total_cost", "test_total_cost",
    "delta_total_cost_test_minus_valid", "valid_fn", "test_fn",
    "valid_recall", "test_recall", "valid_f2", "test_f2",
]]

,candidate_strategy,threshold,valid_total_cost,test_total_cost,delta_total_cost_test_minus_valid,valid_fn,test_fn,valid_recall,test_recall,valid_f2,test_f2
0,drop_high_missing_median,0.19,5100,11260,6160,4,15,0.980,0.960000,0.750383,0.805009
1,baseline_median_all,0.13,5330,11410,6080,5,16,0.975,0.957333,0.762911,0.815909
2,median_all_structural_all,0.31,4960,12860,7900,5,20,0.975,0.946667,0.785657,0.829052


## 与 Day14 未调参结果对比

如果 tuned 方案在 official test 上没有超过 Day14 未调参方案，说明继续追逐 XGBoost 参数的边际价值有限。此时更合理的下一步是 SQL 业务深化、模型解释性分析和 README 最终整理。

In [8]:
day14_path = cfg.metrics_dir / "day14_structural_feature_test_results.csv"
if day14_path.exists():
    day14 = pd.read_csv(day14_path)
    day14[["candidate_group", "threshold", "recall", "f2", "average_precision", "fp", "fn", "total_cost"]]

## Day16 小结模板

- Day16 没有重新调参，也没有重新选 threshold。
- tuned 方案的 official test 成本需要和 Day14 未调参结果对比。
- 如果 valid 低成本没有泛化到 test，应停止继续扩大调参，避免在公开 test 上形成隐性反向选择。
- 本项目下一步更适合进入 SQL 深化、模型解释性分析和最终 README / 报告整理。